# 🚀 Day 1: Environment Setup + SME Data Collection
### **Project:** SME Daily Business Assistant (`SME-Daily-Business`)
### **Assignee:** Deepana Nirmal | **Jira Task:** `KAN-13`
### **Target Models:** Qwen2.5-7B-Instruct & Llama-3-8B-Instruct
### **Hardware:** Google Colab Tesla T4 GPU (15GB VRAM)

---
### 🎯 Objectives for Day 1:
1. Verify Tesla T4 GPU and setup memory safeguards.
2. Mount Google Drive for persistent artifact and model checkpoint storage.
3. Install production-pinned AI/LLM libraries.
4. Establish standard directory structure on Drive and Colab runtime.
5. Download & aggregate raw SME Daily Business domain datasets (~2,500 records).
6. Perform exploratory data analysis (EDA) and save `sme_raw_dataset.jsonl`.

## 1. Mount Google Drive & Hardware Verification

In [ ]:
import os
import sys
import torch

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project'
except Exception:
    PROJECT_ROOT = './AI_SME_Project'

os.makedirs(PROJECT_ROOT, exist_ok=True)

# 2. Verify GPU
print("=== System & Hardware Check ===")
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("⚠️ No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU.")

# 3. Create Project Folders
subfolders = [
    'configs', 'data/raw', 'data/processed', 'data/synthetic', 'data/rag_docs',
    'models/checkpoints', 'models/v1', 'models/v2', 'models/v3', 'models/v4',
    'logs', 'reports', 'vector_db'
]
for sf in subfolders:
    os.makedirs(os.path.join(PROJECT_ROOT, sf), exist_ok=True)

print(f"✅ Project hierarchy verified at: {PROJECT_ROOT}")

## 2. Install Production AI Libraries

In [ ]:
!pip install transformers==4.44.2 datasets==2.20.0 accelerate==0.33.0 peft==0.12.0 bitsandbytes==0.43.3 trl==0.9.6 safetensors==0.4.3 -q
!pip install chromadb==0.5.5 sentence-transformers==3.0.1 evaluate==0.4.2 rouge-score==0.1.2 sacrebleu==2.4.2 tabulate -q
print("✅ Libraries installed successfully!")

## 3. SME Daily Business Data Collection Script
Aggregates domain records across:
- **Customer Support & Invoicing:** Order queries, refund requests, payment follow-ups
- **Financial & Bookkeeping:** Profit calculations, cash flow formulas, SME tax deductions
- **Operational SOPs:** Inventory cycle counts, vendor RFQs, payroll & expense workflows

In [ ]:
import os
import json
import pandas as pd
from datasets import load_dataset

# Ensure PROJECT_ROOT is defined
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project' if os.path.exists('/content/drive/MyDrive') else './AI_SME_Project'

raw_data_dir = os.path.join(PROJECT_ROOT, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)
raw_output_path = os.path.join(raw_data_dir, 'sme_raw_dataset.jsonl')

collected_records = []

# 1. Collect Customer Support & Invoicing (Bitext)
print("1. Fetching Bitext Customer Support & Invoicing dataset from Hugging Face...")
try:
    bitext_ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
    for i, item in enumerate(bitext_ds):
        if i >= 1500:
            break
        collected_records.append({
            "source": "bitext_customer_support",
            "category": item.get("intent", "customer_support"),
            "instruction": item.get("instruction", "").strip(),
            "context": "",
            "response": item.get("response", "").strip()
        })
    print(f"   -> Collected {len(collected_records)} records from Bitext.")
except Exception as e:
    print(f"   -> Notice: Bitext download skipped ({e}).")

# 2. Collect Financial & Bookkeeping QA
print("\n2. Fetching Financial QA dataset from Hugging Face...")
fin_start = len(collected_records)
try:
    fin_ds = load_dataset("virattt/financial-qa-10K", split="train")
    for i, item in enumerate(fin_ds):
        if i >= 1000:
            break
        collected_records.append({
            "source": "financial_qa_10k",
            "category": "financial_accounting",
            "instruction": item.get("question", "").strip(),
            "context": item.get("context", "").strip(),
            "response": item.get("answer", "").strip()
        })
    print(f"   -> Collected {len(collected_records) - fin_start} records from Financial QA.")
except Exception as e:
    print(f"   -> Notice: Financial QA skipped ({e}).")

# 3. Add High-Value Curated SME Daily Business Operational Templates
print("\n3. Adding Curated SME Daily Business Operational SOPs & Templates...")
sme_ops = [
    ("How should an SME draft an invoice overdue notice?", 
     "Subject: Friendly Reminder: Invoice #{inv} Overdue\n\nDear Client,\n\nWe hope you are well. This is a gentle reminder that Invoice #{inv} for the amount of ${amt} was due on {date}. Please let us know once the transfer is initiated or if you need an updated invoice copy.\n\nBest regards,\nFinance Department",
     "billing_invoicing"),
    ("What is the standard procedure for an inventory stock reconciliation?",
     "1. Freeze stock movements during count hours.\n2. Perform physical counts using dual-verifier teams.\n3. Log variances against ERP records.\n4. Investigate variances over 2% threshold.\n5. Post approved stock adjustment journal entries.",
     "inventory_management"),
    ("How do I calculate working capital for my retail business?",
     "Working Capital = Current Assets - Current Liabilities.\nCurrent Assets include cash, inventory, and accounts receivable.\nCurrent Liabilities include accounts payable and short-term debt obligations.\nA positive ratio indicates short-term operational health.",
     "financial_accounting"),
    ("Draft a vendor quote request email for packaging supplies.",
     "Subject: Request for Quotation (RFQ) - Packaging Boxes 2026\n\nDear Supplier,\n\nCould you please provide your best price quote and lead time for 5,000 units of standard corrugated shipping boxes (12x12x12 inches)? Please include bulk discount tiers and delivery terms.\n\nSincerely,\nOperations Lead",
     "procurement_vendor"),
    ("What are the mandatory payroll deductions for small business employees?",
     "Standard payroll deductions include:\n1. Income tax withholding (PAYE / TDS depending on jurisdiction).\n2. Social security / Provident Fund / Pension contributions (both employee and employer matching).\n3. Health / Unemployment insurance contributions.\n4. Voluntary employee deductions (health savings, pension top-ups).",
     "hr_payroll")
]

for idx, (inst, resp, cat) in enumerate(sme_ops):
    for variant in range(20):
        collected_records.append({
            "source": "sme_curated_ops",
            "category": cat,
            "instruction": inst.replace("{inv}", str(1040 + variant)).replace("{amt}", str((variant + 1)*150)).replace("{date}", "2026-09-30"),
            "context": "",
            "response": resp.replace("{inv}", str(1040 + variant)).replace("{amt}", str((variant + 1)*150)).replace("{date}", "2026-09-30")
        })

# Write raw records to JSONL
with open(raw_output_path, 'w', encoding='utf-8') as f:
    for row in collected_records:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f"\n✅ Successfully saved {len(collected_records)} raw records to:\n   {raw_output_path}")

## 4. Exploratory Data Analysis & Validation

In [ ]:
import os
import json
import pandas as pd

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project' if os.path.exists('/content/drive/MyDrive') else './AI_SME_Project'

raw_output_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'sme_raw_dataset.jsonl')
df = pd.read_json(raw_output_path, lines=True)

print("=== 📊 SME Dataset Overview ===")
print(f"Total Collected Records: {len(df):,}")
print("\n--- Records by Source ---")
print(df['source'].value_counts())

print("\n--- Top Categories ---")
print(df['category'].value_counts().head(10))

df['instruction_words'] = df['instruction'].apply(lambda x: len(str(x).split()))
df['response_words'] = df['response'].apply(lambda x: len(str(x).split()))

print("\n--- Word Count Summary ---")
print(df[['instruction_words', 'response_words']].describe())

print("\n--- Sample Record Preview ---")
print(json.dumps(df.iloc[0].to_dict(), indent=2))

## 5. Day 1 Verification & Metadata Save

In [ ]:
import os
import json

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project' if os.path.exists('/content/drive/MyDrive') else './AI_SME_Project'

metadata = {
    "Day": "Day 1 - Environment Setup & Data Collection",
    "Jira_Task": "KAN-13",
    "Engineer": "Deepana Nirmal",
    "Domain": "SME Daily Business",
    "Raw_Dataset_Records": len(df),
    "Raw_Dataset_Path": raw_output_path,
    "T4_Optimizations": {
        "Quantization": "4-bit NF4 with Double Quant",
        "Compute_Dtype": "torch.float16",
        "Adapter_Dtype": "torch.float32"
    },
    "Drive_Synced": True
}

meta_file = os.path.join(PROJECT_ROOT, 'day1_metadata.json')
with open(meta_file, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print("✅ Day 1 Setup Completed and Verified!")
print(json.dumps(metadata, indent=2))
print("\n🎉 Ready for Day 2: Data Cleaning & QLoRA Configuration (KAN-17)!")